In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:
!pip install keras==2.15
!pip install tensorflow==2.15

  Using cached tensorflow-2.15.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.4 kB)
  Using cached ml_dtypes-0.2.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (20 kB)
  Using cached wrapt-1.14.1-cp310-cp310-manylinux_2_5_x86_64.manylinux1_x86_64.manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.7 kB)
  Using cached tensorboard-2.15.2-py3-none-any.whl.metadata (1.7 kB)
  Using cached tensorflow_estimator-2.15.0-py2.py3-none-any.whl.metadata (1.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 475.2/475.2 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 108.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.0/442.0 kB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 8.1 MB/s eta 0:00:00
  Attempting uninstall: wrapt
    Found existing installation: wrapt 1.16.0
    Uninstalling w

In [ ]:
import os
import numpy as np
import cv2
from tensorflow.keras.models import load_model


models = ["best_model"]



for model in models:
  my_model_best = load_model(f"/content/drive/MyDrive/model/{model}.h5")

  datasetNames = ["minirop" ]

  for datasetName in datasetNames:
    dir_test_images=f'/content/drive/MyDrive/{datasetName}/images'

    import glob
    

    dir_test_images = glob.glob(dir_test_images)
    tab_test_images=[]
    tab_files=[]
    names = []
    H = 480
    W = 640

    for fichier in (dir_test_images):
        img=cv2.imread(fichier)
        img=cv2.resize(img, (W, H))
        img=np.array(img, dtype=np.float32)/255
        mask=np.zeros((480, 640, 1), dtype=np.float32)
        prediction=my_model_best.predict(np.array([img]))
        name = fichier.split('/')[-1]
        mask=prediction[0]*255
        mask = cv2.resize(mask, (1280, 960))
        os.makedirs(f"/content/drive/MyDrive/{datasetName}/predicted_{model}" ,exist_ok=True)
        cv2.imwrite(f"/content/drive/MyDrive/{datasetName}/predicted_{model}/{name}", mask)




In [ ]:
import os
import numpy as np
import pandas as pd
from skimage.io import imread
from skimage.transform import resize
from skimage.color import rgb2gray
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score  

def compute_metrics(ground_truth_folder, predicted_folder, output_file):
    gt_images = sorted([f for f in os.listdir(ground_truth_folder) if os.path.isfile(os.path.join(ground_truth_folder, f))])
    pred_images = sorted([f for f in os.listdir(predicted_folder) if os.path.isfile(os.path.join(predicted_folder, f))])

    metrics = []
    i = 0
    for gt_img_name, pred_img_name in zip(gt_images, pred_images):
        gt_image = imread(os.path.join(ground_truth_folder, gt_img_name))
        pred_image = imread(os.path.join(predicted_folder, pred_img_name))

        gt_image = resize(gt_image, (480, 640), anti_aliasing=True)
        pred_image = resize(pred_image, (480, 640), anti_aliasing=True)

        if len(gt_image.shape) == 3:
            gt_image = rgb2gray(gt_image)
        if len(pred_image.shape) == 3:
            pred_image = rgb2gray(pred_image)

        gt_range = (np.min(gt_image), np.max(gt_image))
        pred_range = (np.min(pred_image), np.max(pred_image))
        print(f'Image: {gt_img_name}')
        print(f'Ground Truth Range: {gt_range}, Predicted Range: {pred_range}')

        if i % 3 == 0:
            fig, axes = plt.subplots(1, 2, figsize=(12, 6))
            axes[0].imshow(gt_image, cmap='gray')
            axes[0].set_title(f'Ground Truth Binary: {gt_img_name}')
            axes[0].axis('off')

            axes[1].imshow(pred_image, cmap='gray')
            axes[1].set_title(f'Predicted Binary: {pred_img_name}')
            axes[1].axis('off')

            plt.show()

        gt_image_binary = (gt_image > 0.5).astype(np.int32)
        pred_image_binary = (pred_image > 0.4).astype(np.int32)

        if i % 3 == 0:
            fig, axes = plt.subplots(1, 2, figsize=(12, 6))
            axes[0].imshow(gt_image_binary, cmap='gray')
            axes[0].set_title(f'Ground Truth Binary: {gt_img_name}')
            axes[0].axis('off')

            axes[1].imshow(pred_image_binary, cmap='gray')
            axes[1].set_title(f'Predicted Binary: {pred_img_name}')
            axes[1].axis('off')

            plt.show()

        i += 1

        TP = np.sum((gt_image_binary == 1) & (pred_image_binary == 1))
        TN = np.sum((gt_image_binary == 0) & (pred_image_binary == 0))
        FP = np.sum((gt_image_binary == 0) & (pred_image_binary == 1))
        FN = np.sum((gt_image_binary == 1) & (pred_image_binary == 0))

        accuracy = (TP + TN) / (TP + TN + FP + FN)
        precision = TP / (TP + FP) if TP + FP != 0 else 0
        recall = TP / (TP + FN) if TP + FN != 0 else 0
        f1_score = 2 * (precision * recall) / (precision + recall) if precision + recall != 0 else 0
        iou = TP / (TP + FP + FN) if TP + FP + FN != 0 else 0
        dice = 2 * TP / (2 * TP + FP + FN) if 2 * TP + FP + FN != 0 else 0

        auc_roc = roc_auc_score(gt_image_binary.flatten(), pred_image.flatten())

        metrics.append({
            'image': gt_img_name,
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1_score': f1_score,
            'iou': iou,
            'dice': dice,
            'auc_roc': auc_roc  
        })

        print(f'Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1 Score: {f1_score:.4f}, IOU: {iou:.4f}, Dice: {dice:.4f}, AUC-ROC: {auc_roc:.4f}')
        print('--------------------------------------------------')

    df = pd.DataFrame(metrics)
    df.to_csv(output_file, index=False)
    print(f'Results saved to {output_file}')




datasetNames = [ 'minirop'  ]
models = ["best_model"]

for datasetName in datasetNames:
    print(f"############## {datasetName}  ##############")

    ground_truth_folder = f"/content/drive/MyDrive/{datasetName}/mask"
    predicted_folder = f"/content/drive/MyDrive/{datasetName}/predicted_{models[0]}"
    output_file = f'/content/drive/MyDrive/{datasetName}/test_{models[0]}.csv'
    compute_metrics(ground_truth_folder, predicted_folder, output_file)